# Machine Learning for Applied Research

## 2: Exploratory Data Analysis
In this notebook, we will cover:
- Why EDA comes before modeling, not after
- Visualizing individual features: histograms and boxplots
- Visualizing relationships: scatter plots and a correlation heatmap
- Spotting multicollinearity before it causes problems later

### Introduction

Notebook 1 already took a first pass at the Energy Efficiency data as part of framing the problem, namely checking for missing values and computing correlations with the target. This notebook goes a step further with visual exploration: the kind of look worth giving any new dataset before training a single model, since plots catch problems (such as skew, unexpected gaps, redundant features) that summary statistics alone can hide.

In [ ]:
!pip install ucimlrepo --quiet

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

energy = fetch_ucirepo(id=242)
X = energy.data.features
y = energy.data.targets

X.columns = ["Relative Compactness",
             "Surface Area",
             "Wall Area",
             "Roof Area",
             "Overall Height",
             "Orientation",
             "Glazing Area",
             "Glazing Area Distribution"]

y.columns = ["Heating Load", "Cooling Load"]

target_col = y.columns[0]   # Heating Load
df = pd.concat([X, y[[target_col]]], axis=1)

print(f"Shape: {df.shape}")

### Visualizing Individual Features

In [ ]:
df.hist(figsize=(12, 10), bins=15)
plt.tight_layout()
plt.show()

Notice that several features take only a handful of distinct values rather than a smooth continuous spread: the histograms look like a small number of spikes, not a bell curve. This is a designed experiment (a set of building shapes varied systematically), not organic measurement data, which is worth knowing before you start modeling.

#### 1.1 Exercises:
> (a) Plot a histogram of `Glazing Area` (`bins=10`) and describe its shape in one sentence.
>
> (b) Plot a boxplot of `Surface Area`. Are there any visible outliers?
>
> (c) Given what you saw in (a), would you be comfortable applying a log-transform (or anything else requiring strictly positive values) to `Glazing Area`? Why or why not?

In [ ]:
# Your solution:




##### Solution:


In [ ]:
plt.figure(figsize=(6, 4))
df["Glazing Area"].hist(bins=10)
plt.xlabel("Glazing Area")
plt.title("Distribution of Glazing Area")
plt.show()

plt.figure(figsize=(6, 4))
plt.boxplot(df["Surface Area"])
plt.title("Surface Area")
plt.show()

# (a) Rather than a smooth spread, the histogram shows a handful of
# distinct spikes. Glazing Area only takes a few fixed values in this
# designed experiment, including 0.

# (b) Surface Area's boxplot typically shows no extreme outliers here.
# Values cluster into the same small number of designed levels as the
# other features, without any single point sitting far outside the rest.

# (c) No. Glazing Area includes 0 in this dataset, and a log transform is
# undefined at zero. This is the kind of problem a quick
# histogram catches early, before it turns into a runtime error or a
# silent NaN buried inside a larger pipeline.

### Visualizing Relationships and Spotting Multicollinearity

A correlation heatmap extends Notebook 1's single-column correlation check to every pair of features at once, which is useful for spotting **multicollinearity**: pairs of features that are so strongly related to each other that they carry largely redundant information.

In [ ]:
corr_matrix = df.corr(numeric_only=True)

plt.figure(figsize=(8, 7))
plt.imshow(corr_matrix, cmap="coolwarm", vmin=-1, vmax=1)
plt.colorbar(label="Correlation")
plt.xticks(range(len(corr_matrix.columns)), corr_matrix.columns, rotation=90)
plt.yticks(range(len(corr_matrix.columns)), corr_matrix.columns)
plt.title("Correlation heatmap")
plt.tight_layout()
plt.show()

Look for any off-diagonal cell close to dark red (+1) or dark blue (-1). We will discuss a safeguard for near-perfect relationships later.

#### 1.2 Exercises:
> (a) From `corr_matrix`, excluding the diagonal, find the pair of features with the strongest correlation (positive or negative, with the `abs()` method). Then `unstack` the matrix and sort the values (`ascending=False`). This will list the diagonal (`1.0`) first, you can ignore those.
>
> (b) Create a scatter plot of these two features against each other.
>
> (c) If you were about to fit a linear regression, would you be tempted to include both of these features as-is?

In [ ]:
# Your solution:




##### Solution:


In [ ]:
correlation_pairs = corr_matrix.abs().unstack().sort_values(ascending=False)
print(correlation_pairs.head(10))   # the first several entries are the diagonal (1.0)

# (b) Once you've identified the top non-diagonal pair (commonly
# Relative Compactness and Surface Area):
plt.figure(figsize=(6, 5))
plt.scatter(df["Relative Compactness"], df["Surface Area"], s=10, alpha=0.5)
plt.xlabel("Relative Compactness")
plt.ylabel("Surface Area")
plt.title("A near-perfectly correlated feature pair")
plt.show()

# (c) Including both would mean feeding the model two versions of nearly
# the same information, which will make raw linear regresssion
# coefficients hard to interpret cleanly.

### Conclusion

A short notebook with a large payoff:

- **Histograms and boxplots** catch skew, designed-experiment structure, and problematic values (like Glazing Area's zeros) before they cause silent bugs later
- **A correlation heatmap** flags multicollinearity early
- None of this takes long! Just a few minutes of visual exploration can save hours of confused debugging once modeling starts
